Batch matching theory
* In our V1 engine, if 100 people bought a $10 coffee, the bank statement showed 100 individual deposits of $9.68 (after the Stripe fee).
* In the real world, Stripe doesn't do that. It waits until midnight, scoops up all 100 transactions, calculates the total ($1,000), subtracts the total fees (2.9% of $1,000 + 100 * $0.30 = $59.00), and sends exactly one deposit to your bank account the next day for $941.00.

This means our new schemas will look completely different:
* Internal Ledger: Still has individual Transaction_IDs, precise Dates (down to the second), and individual Amounts.
* Bank Statement: No longer has Transaction_IDs! Instead, it has a Batch_Date, a Total_Settled_Amount, and a generic Description (like "STRIPE PAYOUT").

In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os

fake = Faker()
Faker.seed(100)
np.random.seed(100)
random.seed(100)

def generate_batched_data(num_records=5000):
    print(f"Generating {num_records} internal ledger records...")
    
    # Generating a mock internal ledger with transaction-level data (one row per transaction)
    internal_data = []
    # Let's generate data over a specific 5-day period so we have distinct daily batches
    for _ in range(num_records):
        internal_data.append({
            "Transaction_ID": fake.uuid4()[:16].upper(),
            "Date": fake.date_time_between(start_date="-5d", end_date="now").replace(microsecond=0),
            "Customer_Name": fake.name(),
            "Amount": round(random.uniform(15.0, 150.0), 2),
            "Gateway": "Stripe"
        })
    
    internal_df = pd.DataFrame(internal_data)

    # Generating a bank statement that simulates the end-of-day payouts from Stripe, which will be one row per day
    print("Simulating end-of-day gateway batching...")
    
    # Extract just the Date (strip away the hours/minutes) so we can group by day
    internal_df['Date'] = pd.to_datetime(internal_df['Date']).dt.date
    
    # Group by the day and calculate the metrics we need for the payout
    # We need the sum of all money, and the count of transactions to calculate the $0.30 flat fees
    # When you tell pandas to group data by a specific column (like Just_Date), pandas takes that column and turns it into the Index (the "address" or row label) of the new, summarized table.
    # Calling .reset_index() tells pandas: "Take whatever is currently acting as the Index, push it back into the table as a normal column, 
    # and give me back the standard 0, 1, 2, 3 row numbers."
    daily_batches = internal_df.groupby('Date').agg(
        Total_Gross_Amount=('Amount', 'sum'),
        Transaction_Count=('Amount', 'count')
    ).reset_index()
    
    # Calculate the real-world batch payout (Stripe takes 2.9% of the total, plus $0.30 PER transaction)
    daily_batches['Total_Fees'] = (daily_batches['Total_Gross_Amount'] * 0.029) + (daily_batches['Transaction_Count'] * 0.30)
    daily_batches['Total_Settled_Amount'] = round(daily_batches['Total_Gross_Amount'] - daily_batches['Total_Fees'], 2)
    
    # Build the final Bank Statement format
    bank_df = pd.DataFrame({
        # Simulate a T+2 settlement delay for the entire batch
        "Settlement_Date": pd.to_datetime(daily_batches['Date']) + pd.to_timedelta(2, unit='D'),
        "Description": "STRIPE PAYOUT",
        "Total_Settled_Amount": daily_batches['Total_Settled_Amount']
    })

    internal_df = internal_df.drop(columns=['Date'])

    # Export the data to CSV files
    os.makedirs('../data', exist_ok=True)
    internal_df.to_csv('../data/Batched_Internal_Ledger.csv', index=False)
    bank_df.to_csv('../data/Batched_Bank_Statement.csv', index=False)
    
    print("Success! Batched data generated.")
    print(f"\nSneak Peek - Bank Statement:\n{bank_df.head()}")

# if __name__ == "__main__":
#     generate_batched_data()

In [5]:
generate_batched_data()

Generating 5000 internal ledger records...
Simulating end-of-day gateway batching...
Success! Batched data generated.

Sneak Peek - Bank Statement:
  Settlement_Date    Description  Total_Settled_Amount
0      2026-03-14  STRIPE PAYOUT              25398.82
1      2026-03-15  STRIPE PAYOUT              80890.19
2      2026-03-16  STRIPE PAYOUT              81340.92
3      2026-03-17  STRIPE PAYOUT              79657.39
4      2026-03-18  STRIPE PAYOUT              78780.82


Aggregate Matching
* Mathematically recreate the payment gateway's grouping logic on our own internal data, calculate what we expect the bank to pay us for that day, and then compare our expectations against reality.

In [ ]:
import pandas as pd
import numpy as np

def run_batched_reconciliation():
    print("--- Starting V2 Engine: Batched Reconciliation ---")
    internal_df = pd.read_csv('../data/Batched_Internal_Ledger.csv')
    bank_df = pd.read_csv('../data/Batched_Bank_Statement.csv')
    internal_df['Date'] = pd.to_datetime(internal_df['Date'])
    bank_df['Settlement_Date'] = pd.to_datetime(bank_df['Settlement_Date'])
    
    print("Aggregating internal transactions by day...")
    internal_df['Date'] = internal_df['Date'].dt.date
    
    internal_daily = internal_df.groupby('Date').agg(
        Total_Gross_Amount=('Amount', 'sum'),
        Transaction_Count=('Amount', 'count')
    ).reset_index()
    
    expected_fees = (internal_daily['Total_Gross_Amount'] * 0.029) + (internal_daily['Transaction_Count'] * 0.30)
    internal_daily['Expected_Settlement'] = round(internal_daily['Total_Gross_Amount'] - expected_fees, 2)
    
    internal_daily['Expected_Settlement_Date'] = pd.to_datetime(internal_daily['Date']) + pd.to_timedelta(2, unit='D')

    # Matching
    print("Matching expected batches against actual bank deposits...")
    batched_match = pd.merge(
        internal_daily,
        bank_df,
        left_on='Expected_Settlement_Date',
        right_on='Settlement_Date',
        how='left' 
    )
    
    # Validation
    batched_match['Is_Perfect_Match'] = np.isclose(
        batched_match['Expected_Settlement'], 
        batched_match['Total_Settled_Amount'], 
        atol=0.01
    )
    print("\n--- Final Batched Reconciliation Report ---")
    
    display_cols = ['Date', 'Total_Gross_Amount', 'Expected_Settlement', 'Total_Settled_Amount', 'Is_Perfect_Match']
    print(batched_match[display_cols].to_string(index=False))
    
    return batched_match

if __name__ == "__main__":
    run_batched_reconciliation()